# Footy AI - train the pitch keypoint model and the ball detector

Two Ultralytics models for the `football_summary` pipeline:

1. **Ball detector** - a single-class YOLO detection model trained at 1280 px so a ball of 8-15 px is still visible. Replaces the weak ball class of `models/best.pt`.
2. **Pitch keypoint model** - a YOLO *pose* model that predicts the pitch line intersections (32 keypoints in the Roboflow football-field dataset). Its output gives a homography from image to pitch coordinates, which is what a real penalty area, goal line and wing zone need.

Runtime: **GPU** (Colab: Runtime > Change runtime type > T4 or better). A T4 trains the ball model in roughly 2-3 hours for 100 epochs and the keypoint model in about 1 hour. An L4/A100 is faster and lets you raise the batch size.

Fill in the dataset fields in the *Datasets* cell; the API key is asked for interactively and never written to the notebook.

In [ ]:
!nvidia-smi
!pip install -q "ultralytics>=8.3" roboflow
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "torch", torch.__version__, "cuda", torch.cuda.is_available())

## Optional: keep the weights on Google Drive

Colab wipes `/content` when the runtime ends. Mount Drive so `best.pt` survives; the final cell copies weights there.

In [ ]:
SAVE_TO_DRIVE = True  # set False to skip
DRIVE_DIR = '/content/drive/MyDrive/footy_ai_models'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs(DRIVE_DIR, exist_ok=True)

## Datasets

Both datasets are downloaded through the Roboflow SDK. Open each dataset on Roboflow Universe, click **Download Dataset**, pick the format below, choose *show download code*, and copy the workspace, project and version into the fields:

| Model | Roboflow export format | Notes |
| --- | --- | --- |
| Ball detector | **YOLOv8** (also loads in YOLO11) | keep images at native size or *Fit within 1280x1280*; never 640, the ball disappears |
| Pitch keypoints | **YOLOv8** keypoint export | 640 px is fine; the pitch is the whole frame |

If your datasets are not on Roboflow, upload a zip in YOLO layout (`train/images`, `train/labels`, `valid/...`, `data.yaml`) instead and set `BALL_DATA_YAML` / `PITCH_DATA_YAML` to the yaml paths.

In [ ]:
from getpass import getpass
ROBOFLOW_API_KEY = getpass('Roboflow API key (input hidden): ')

# --- ball detector dataset (replace with the one you found) ---
BALL_WORKSPACE = 'roboflow-jvuqo'
BALL_PROJECT = 'football-ball-detection-rejhg'
BALL_VERSION = 2

# --- pitch keypoint dataset (replace with the one you found) ---
PITCH_WORKSPACE = 'roboflow-jvuqo'
PITCH_PROJECT = 'football-field-detection-f07vi'
PITCH_VERSION = 14

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

ball_ds = rf.workspace(BALL_WORKSPACE).project(BALL_PROJECT).version(BALL_VERSION).download('yolov8', location='/content/datasets/ball')
pitch_ds = rf.workspace(PITCH_WORKSPACE).project(PITCH_PROJECT).version(PITCH_VERSION).download('yolov8', location='/content/datasets/pitch')

BALL_DATA_YAML = f'{ball_ds.location}/data.yaml'
PITCH_DATA_YAML = f'{pitch_ds.location}/data.yaml'
print(BALL_DATA_YAML)
print(PITCH_DATA_YAML)

In [ ]:
# Roboflow yaml files sometimes point at '../train/images'. Pin them to absolute paths
# and print the class list and keypoint shape so a wrong export is caught here,
# not after an hour of training.
import pathlib
import yaml


def pin(yaml_path):
    path = pathlib.Path(yaml_path)
    data = yaml.safe_load(path.read_text())
    root = path.parent
    for split in ('train', 'val', 'test'):
        candidate = root / ('valid' if split == 'val' else split) / 'images'
        if split in data and candidate.exists():
            data[split] = str(candidate)
    data['path'] = str(root)
    path.write_text(yaml.safe_dump(data))
    return data


ball_cfg = pin(BALL_DATA_YAML)
pitch_cfg = pin(PITCH_DATA_YAML)
print('ball classes:', ball_cfg.get('names'))
print('pitch classes:', pitch_cfg.get('names'), 'kpt_shape:', pitch_cfg.get('kpt_shape'), 'flip_idx present:', 'flip_idx' in pitch_cfg)
assert 'kpt_shape' in pitch_cfg, 'The pitch dataset must be a keypoint export (kpt_shape missing).'

## 1. Ball detector

Small object, so: **imgsz 1280**, a medium model (`yolo11m`), mosaic kept on (it multiplies the tiny-object examples), `close_mosaic=10` to finish on clean images. Lower `batch` to 4 if the GPU runs out of memory; raise to 16 on an A100.

In [ ]:
from ultralytics import YOLO

ball_model = YOLO('yolo11m.pt')
ball_model.train(
    data=BALL_DATA_YAML,
    epochs=100,
    imgsz=1280,
    batch=8,
    patience=25,
    optimizer='auto',
    close_mosaic=10,
    degrees=0.0,   # broadcast frames are not rotated
    fliplr=0.5,
    scale=0.5,
    mixup=0.1,
    project='/content/runs',
    name='ball_detector',
    exist_ok=True,
    plots=True,
)
BALL_BEST = '/content/runs/ball_detector/weights/best.pt'

In [ ]:
# Validate at the training size and at the 960 px the pipeline uses for 720p sources.
for size in (1280, 960):
    metrics = YOLO(BALL_BEST).val(data=BALL_DATA_YAML, imgsz=size, batch=8, plots=False)
    print(f'imgsz={size}: mAP50={metrics.box.map50:.3f} mAP50-95={metrics.box.map:.3f} recall={metrics.box.mr:.3f}')

## 2. Pitch keypoint model

A pose model. The pitch is one object per frame with 32 keypoints, most of them off-screen at any moment (labelled invisible). Settings follow the Roboflow sports recipe: **mosaic off** (four pitch quarters stitched together is not a pitch), **no horizontal flip** unless the dataset yaml carries `flip_idx` (flipping swaps left and right keypoints), 640 px.

In [ ]:
pitch_model = YOLO('yolo11m-pose.pt')
pitch_model.train(
    data=PITCH_DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=25,
    mosaic=0.0,
    fliplr=0.5 if 'flip_idx' in pitch_cfg else 0.0,
    degrees=0.0,
    scale=0.3,
    project='/content/runs',
    name='pitch_keypoints',
    exist_ok=True,
    plots=True,
)
PITCH_BEST = '/content/runs/pitch_keypoints/weights/best.pt'
metrics = YOLO(PITCH_BEST).val(data=PITCH_DATA_YAML, imgsz=640, plots=False)
print(f'pose mAP50={metrics.pose.map50:.3f} mAP50-95={metrics.pose.map:.3f}')

## 3. Smoke test on a real frame

Upload one frame from your own footage (720p or 1080p) and check that the ball is boxed and the keypoints land on line intersections. If the ball is missed here, train longer or add labelled frames from your own videos to the dataset.

In [ ]:
from google.colab import files
import cv2
from IPython.display import Image, display

uploaded = files.upload()
for name in uploaded:
    ball_pred = YOLO(BALL_BEST).predict(name, imgsz=1280, conf=0.15, verbose=False)[0]
    pitch_pred = YOLO(PITCH_BEST).predict(name, imgsz=640, conf=0.3, verbose=False)[0]
    frame = pitch_pred.plot(boxes=False)
    for box in ball_pred.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imwrite('smoke.jpg', frame)
    display(Image('smoke.jpg', width=900))
    visible = 0
    if pitch_pred.keypoints is not None and len(pitch_pred.keypoints):
        visible = int((pitch_pred.keypoints.conf[0] > 0.5).sum())
    print(name, 'balls:', len(ball_pred.boxes), 'keypoints visible:', visible)

## 4. Save the weights

Copy to Drive (if mounted) and download. In the `football_summary` repository place them as:

- `models/ball_best.pt`
- `models/pitch_keypoints_best.pt`

The pipeline does not consume them yet; wiring them in is the next step (a ball-model option in `trackers/tracker.py` and a homography module that turns the keypoints into pitch coordinates for `track_events/`).

In [ ]:
import os
import shutil

targets = {'ball_best.pt': BALL_BEST, 'pitch_keypoints_best.pt': PITCH_BEST}
for out_name, src in targets.items():
    shutil.copy(src, f'/content/{out_name}')
    if SAVE_TO_DRIVE:
        shutil.copy(src, os.path.join(DRIVE_DIR, out_name))
        print('saved to Drive:', os.path.join(DRIVE_DIR, out_name))
from google.colab import files
for out_name in targets:
    files.download(f'/content/{out_name}')